# Consumer AI subscriptions on e-receipts: a step-by-step walkthrough

This notebook shows how the `ai_consumer` job (branch `feat/ai-consumer-subs`) measures gross adds, churn and subscribers for each AI provider, using the tables its local run wrote to `SRS_MCP.AI_CONSUMER`. It also checks the meeting suggestion that NIQ's `CANCELLATION_ITEMS` feed holds cancellation emails we don't already read.

1. **Cancellation feed check:** `CANCELLATION_ITEMS` against `ITEMS_EXTENDED`.
2. **Receipts we classify:** what each provider's emails look like.
3. **Gross adds, step by step, per provider.**
4. **Churn, step by step, per provider:** observed lapses, then the churn rates the model uses.
5. **Subscribers:** the stock-flow index, reconciled month by month.

Set `REGION` to `"us"` or `"eu"`. Queries run on `ANALYST_L`.

In [1]:
import pandas as pd, numpy as np, snowflake.connector as sc
from IPython.display import display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)
con = sc.connect(connection_name="default"); con.cursor().execute("USE WAREHOUSE ANALYST_L")

def q(sql):
    cur = con.cursor(); cur.execute(sql); df = cur.fetch_pandas_all()
    df.columns = [c.lower() for c in df.columns]
    return df

REGION = "us"
COUNTRY = "us" if REGION == "us" else "eu"
WINDOWS = pd.read_csv("../../../srs-ai-consumer/workflows/jobs/ai/consumer/artifacts/engine/params/lapse_windows.csv")
DATA_END = pd.Timestamp(q(f"SELECT MAX(dt) AS d FROM srs_mcp.ai_consumer.receipt_events WHERE region = '{REGION}'").d[0])
print("data end:", DATA_END.date())
WINDOWS[WINDOWS.region == REGION][["prov", "channel", "capture_all", "W_monthly", "W_source", "rate_source"]]

data end: 2026-10-20


,prov,channel,capture_all,W_monthly,W_source,rate_source
15,chatgpt,apple,0.9394,42,own cell,reliable_subset
16,chatgpt,google_play,0.9863,40,own cell,full_panel
17,claude,apple,0.9653,43,own cell,reliable_subset
18,claude,direct,0.9872,41,own cell,full_panel
19,claude,google_play,0.9766,42,own cell,full_panel
20,gemini,apple,0.9485,44,own cell,reliable_subset
21,gemini,google_play,0.9855,40,own cell,full_panel
22,grok,apple,0.9333,60,own cell,reliable_subset
23,grok,google_play,0.9544,91,own cell,reliable_subset
24,perplexity,apple,0.9354,59,own cell,reliable_subset


## 1. Does `CANCELLATION_ITEMS` add cancellation emails?

The question from the 20 Oct meeting was whether NIQ keeps most cancellations in a separate `SHARE_NIQ.SRS.CANCELLATION_ITEMS` view rather than in `ITEMS_EXTENDED`, the receipt table behind `srs_slice.common.items_v` that the job reads.

We match the two views on `item_id` for the cancellation senders that carry AI items:

| merchant_id | Sender |
|---|---|
| 20122 | iTunes Cancellation |
| 3452 | Google Play Cancellation |
| 20191 | Google One Cancellation |
| 4078 | Chat GPT Cancellation |
| 20139 | Anthropic Cancellation |
| 20140 | Cursor Cancellation |
| 4213 | Perplexity AI Cancellation |

In [2]:
IDS = "20122, 3452, 20191, 4078, 20139, 20140, 4213"
overlap = q(f'''
WITH c AS (SELECT item_id, merchant_id, merchant_name FROM SHARE_NIQ.SRS.CANCELLATION_ITEMS
           WHERE merchant_id IN ({IDS}) AND order_date >= '2025-10-01'),
     e AS (SELECT item_id, merchant_id, merchant_name FROM SHARE_NIQ.SRS.ITEMS_EXTENDED
           WHERE merchant_id IN ({IDS}) AND order_date >= '2025-10-01')
SELECT COALESCE(c.merchant_id, e.merchant_id) AS merchant_id,
       COALESCE(ANY_VALUE(c.merchant_name), ANY_VALUE(e.merchant_name)) AS sender,
       COUNT(c.item_id) AS in_cancellation_items, COUNT(e.item_id) AS in_items_extended,
       COUNT_IF(c.item_id IS NOT NULL AND e.item_id IS NOT NULL) AS in_both,
       COUNT_IF(e.item_id IS NULL) AS only_in_cancellation_items
FROM c FULL OUTER JOIN e ON c.item_id = e.item_id
GROUP BY 1 ORDER BY 1''')
overlap["share_of_ci_already_read"] = (overlap.in_both / overlap.in_cancellation_items.replace(0, np.nan)).round(4)
overlap

,merchant_id,sender,in_cancellation_items,in_items_extended,in_both,only_in_cancellation_items,share_of_ci_already_read
0,3452,Google Play Cancellation,2017559,2039567,2002714,14845,0.9926
1,4078,Chat GPT Cancellation,119888,121794,119706,182,0.9985
2,4213,Perplexity AI Cancellation,0,21281,0,0,NaN
3,20122,iTunes Cancellation,2145540,2174240,2142001,3539,0.9984
4,20139,Anthropic Cancellation,34012,34759,33951,61,0.9982
5,20140,Cursor Cancellation,1181,1192,1181,0,1.0000
6,20191,Google One Cancellation,229142,231664,227874,1268,0.9945


**Reading it:** almost every `CANCELLATION_ITEMS` row (October 2025 onward) is already in `ITEMS_EXTENDED`, so we already read it. Perplexity AI Cancellation (4213) has no rows in `CANCELLATION_ITEMS`, but we read it from `ITEMS_EXTENDED`.

Next: are the few rows found only in `CANCELLATION_ITEMS` AI cancellations, and are they in mailboxes still in the panel?

In [3]:
extra = q(f'''
WITH c AS (SELECT item_id, mailbox_id, merchant_id FROM SHARE_NIQ.SRS.CANCELLATION_ITEMS
           WHERE merchant_id IN ({IDS}) AND order_date >= '2022-01-01'
             AND (merchant_name || ' ' || COALESCE(description, '')) ILIKE ANY ('%chatgpt%', '%chat gpt%', '%openai%', '%anthropic%',
                 '%claude%', '%google ai%', '%gemini%', '%ai premium%', '%ai pro%', '%grok%', '%x premium%', '%premium+%', '%muse%',
                 '%power plan%', '%perplexity%', '%cursor%')),
     e AS (SELECT item_id FROM SHARE_NIQ.SRS.ITEMS_EXTENDED WHERE merchant_id IN ({IDS}) AND order_date >= '2021-12-01'),
     act AS (SELECT mailbox_id FROM SHARE_NIQ.SRS.MAILBOXES_ENHANCED WHERE status = 'ACTIVE')
SELECT c.merchant_id, COUNT(*) AS ai_cancel_rows, COUNT_IF(e.item_id IS NULL) AS only_in_cancellation_items,
       COUNT_IF(e.item_id IS NULL AND c.mailbox_id IN (SELECT mailbox_id FROM act)) AS only_in_ci_active_mailbox
FROM c LEFT JOIN e ON c.item_id = e.item_id
GROUP BY 1 ORDER BY 1''')
extra.loc["total"] = extra.sum(numeric_only=True)
extra

,merchant_id,ai_cancel_rows,only_in_cancellation_items,only_in_ci_active_mailbox
0,3452,96523,385,17
1,4078,206715,289,25
2,20122,237711,362,18
3,20139,47444,84,7
4,20140,3329,1,0
5,20191,119899,497,7
total,88122,711621,1618,74


**Conclusion:** `CANCELLATION_ITEMS` adds essentially nothing for these providers. The rows missing from `ITEMS_EXTENDED` almost all come from mailboxes that have left the panel, which the panel rules exclude anyway. Only a few dozen AI cancellation rows since 2022 sit in active mailboxes. Cancellation coverage therefore stays where Phase 1 measured it (no provider × channel reaches 95%), and churn stays lapse-based.

## 2. The receipts we classify

`receipt_events` holds one row per kept paid receipt (`kind = 'charge'`) or cancellation email (`kind = 'cancel'`). Each row carries its provider, plan (tier × cadence) and billing channel.

Direct ChatGPT paid receipts are kept with `reference_only = TRUE`: they are sign-ups only, with no renewals, so they never form subscriptions. Below are the last 12 months by provider × channel × kind.

In [4]:
mix = q(f'''
SELECT provider, channel, kind, COUNT(*) AS rows_, COUNT(DISTINCT uid) AS members
FROM srs_mcp.ai_consumer.receipt_events
WHERE region = '{REGION}' AND dt > DATEADD(month, -12, '{DATA_END.date()}') AND NOT reference_only
GROUP BY ALL ORDER BY provider, channel, kind''')
mix.pivot_table(index=["provider", "channel"], columns="kind", values=["rows_", "members"], aggfunc="sum").fillna(0).astype(int)

members         rows_        
kind                    cancel charge cancel  charge
provider   channel                                  
chatgpt    apple         49798  96148  67619  544426
           google_play   13204  23669  23090  126540
claude     apple          7240  17230   8672   58957
           direct        24187  69677  29983  327045
           google_play    1940   3238   3736   12536
gemini     apple          2510   4723   2743   16581
           google_play   41465  80438  74537  429665
grok       apple         11361   9087  14472   26975
           google_play    6069   3274   9908    9710
muse       apple            34    289     34     310
           google_play      13    151     16     157
perplexity apple          2682   3786   2989   11817
           direct         7559      0   8167       0
           google_play     369    510    593    1924
           other             0     46      0     124

In [4]:
# A few recent raw receipt lines from items_v behind the classification (US only).
if REGION == "us":
    display(q('''
    SELECT merchant_name, description, item_price, order_date
    FROM srs_slice.common.items_v
    WHERE merchant_id IN (5, 20122, 1493, 3452, 4070, 4078, 20127, 20139, 20187, 20191, 4212, 4213)
      AND order_date >= DATEADD(day, -14, CURRENT_DATE)
      AND description ILIKE ANY ('%chatgpt%', '%claude%', '%google ai%', '%supergrok%', '%perplexity%', '%muse%', '%power plan%')
    QUALIFY ROW_NUMBER() OVER (PARTITION BY merchant_name, LEFT(description, 25) ORDER BY order_date DESC) = 1
    ORDER BY merchant_name LIMIT 40'''))

,merchant_name,description,item_price,order_date
0,Anthropic Cancellation,Claude Pro,NaN,2026-10-20
1,Anthropic Cancellation,Claude Max,NaN,2026-10-20
2,Chat GPT,ChatGPT Business Subscription,50.00,2026-10-19
3,Chat GPT,ChatGPT Go Subscription,8.00,2026-10-19
4,Chat GPT,ChatGPT Pro 200,200.00,2026-10-15
5,Chat GPT,ChatGPT Pro 100,100.00,2026-10-19
6,Chat GPT,ChatGPT Plus Subscription,20.00,2026-10-20
7,Chat GPT Cancellation,ChatGPT Free subscription cancellation,NaN,2026-10-20
8,Chat GPT Cancellation,ChatGPT Pro 100 subscription cancellation,NaN,2026-10-20
9,Chat GPT Cancellation,ChatGPT Plus subscription cancellation,NaN,2026-10-20


## 3 and 4. Gross adds and churn, step by step

For each provider, `walkthrough(provider)` reproduces the rules from `FINAL_SPEC.md` in plain pandas.

**Gross adds**
1. Take the provider's paid receipts (`kind = 'charge'`, `reference_only = FALSE`), one per member per day.
2. Sort each member's receipts by date. A new subscription starts at the first receipt, and again whenever the gap since the previous receipt is longer than the lapse window `W`.
   - `W` is the 99th-percentile renewal gap for that provider × channel (`lapse_windows.csv`; 135 days for quarterly plans, 400 for annual).
3. **Gross add** = the first receipt of each subscription, counted in the month it falls in.

**Churn (observed)**
4. A subscription ends when no receipt arrives within `W` of its last payment. The churn is dated when it becomes known: last payment + `W`.
   - Subscriptions whose window hasn't passed by the data end are still live.
5. **Observed churn rate** = churns detected in the month ÷ subscriptions live at the start of the month.

The model differs from this simplified version in four ways:
- Churn **rates** come only from members whose renewals we see reliably: the full panel on channels with ≥ 97.5% capture, and the reliable subset (another regular subscription email in the same mailbox that month) for Apple and the rest. See the `churn_rates` table.
- The rates are applied to all subscriptions by plan and tenure.
- It handles mailbox exits and held months.
- A cancel followed by a later payment counts as churn plus a gross add.

So the simple counts below should be close to, but not exactly equal to, the published `subs_monthly` (`pub_*` columns). What to look for:

- **Gross adds:** published is a few percent higher. It also counts a re-subscription after a cancellation as a new add, and uses mailbox history the simple version drops.
- **Churn rate:** for Apple-heavy providers (ChatGPT, Grok, Perplexity) the simple rate is higher than published. A missed Apple renewal email looks like churn in the simple version. The model avoids this by taking Apple's rate only from the reliable subset, which is the reason that subset exists.
- **Claude and Gemini:** mostly direct and Google Play, where nearly every renewal is captured, so simple and published churn are close (published a little higher, because it adds the cancel-then-pay churns).
- **`pub_subs` vs `live_end`:** `live_end` is the observed count, equal to `pub_subs_observed`. `pub_subs` is the modelled count, which adds back Apple subscribers wrongly treated as churned.

In [6]:
def load_charges(provider):
    ev = q(f'''
    SELECT uid, dt, channel, plan, cadence
    FROM srs_mcp.ai_consumer.receipt_events
    WHERE region = '{REGION}' AND provider = '{provider}' AND kind = 'charge' AND NOT reference_only
    QUALIFY ROW_NUMBER() OVER (PARTITION BY uid, dt ORDER BY channel) = 1''')
    ev["dt"] = pd.to_datetime(ev.dt)
    w = WINDOWS[(WINDOWS.region == REGION) & (WINDOWS.prov == provider)].set_index("channel").W_monthly
    ev["W"] = np.select([ev.cadence == "A", ev.cadence == "Q"], [400, 135], ev.channel.map(w).fillna(62)).astype(int)
    return ev.sort_values(["uid", "dt"]).reset_index(drop=True)

def build_subs(ev):
    gap = ev.groupby("uid").dt.diff().dt.days
    prev_W = ev.groupby("uid").W.shift()
    ev["new_sub"] = gap.isna() | (gap > prev_W)                      # step 2
    ev["sub_id"] = ev.new_sub.cumsum()
    s = ev.groupby("sub_id").agg(uid=("uid", "first"), start=("dt", "first"), last_pay=("dt", "last"),
                                 n_pay=("dt", "size"), channel=("channel", "first"), plan=("plan", "first"),
                                 W=("W", "last"))
    s["detect"] = s.last_pay + pd.to_timedelta(s.W, unit="D")      # step 4
    s["churned"] = s.detect <= DATA_END
    return ev, s

def monthly(s):
    out = []
    for m in pd.period_range("2024-01", DATA_END.to_period("M"), freq="M"):
        ms, me = m.start_time, min(m.end_time.normalize(), DATA_END)
        live_start = ((s.start < ms) & (~s.churned | (s.detect >= ms))).sum()
        out.append(dict(month=str(m),
                        gross_adds=((s.start >= ms) & (s.start <= me)).sum(),             # step 3
                        churn=(s.churned & (s.detect >= ms) & (s.detect <= me)).sum(),    # step 4
                        live_start=live_start,
                        live_end=((s.start <= me) & (~s.churned | (s.detect > me))).sum()))
    d = pd.DataFrame(out)
    d["churn_rate"] = (d.churn / d.live_start.replace(0, np.nan)).round(4)                  # step 5
    return d

def published(provider):
    return q(f'''
    SELECT TO_VARCHAR(month, 'YYYY-MM') AS month, gross_adds AS pub_gross_adds, ROUND(churn) AS pub_churn,
           ROUND(churn_rate, 4) AS pub_churn_rate, ROUND(subs) AS pub_subs, subs_observed AS pub_subs_observed, held
    FROM srs_mcp.ai_consumer.subs_monthly
    WHERE region = '{REGION}' AND country = '{COUNTRY}' AND provider = '{provider}' AND plan = 'all'
    ORDER BY 1''')

def rates(provider):
    return q(f'''
    SELECT TO_VARCHAR(month, 'YYYY-MM') AS month, channel, rate_source, SUM(active) AS active, SUM(churned) AS churned,
           ROUND(SUM(churned) / NULLIF(SUM(active), 0), 4) AS rate
    FROM srs_mcp.ai_consumer.churn_rates
    WHERE region = '{REGION}' AND provider = '{provider}' AND month >= DATEADD(month, -3, '{DATA_END.date()}')
    GROUP BY ALL ORDER BY month, channel''')

def walkthrough(provider, example_rows=12):
    ev = load_charges(provider)
    ev, s = build_subs(ev)
    print(f"{provider}: {len(ev):,} paid receipts, {ev.uid.nunique():,} members, {len(s):,} subscriptions")
    print("Lapse window W (days) by channel x cadence:", ev.groupby(["channel", "cadence"]).W.first().to_dict())
    many = s[s.n_pay >= 3].groupby("uid").size()
    ex_uid = many[many >= 2].index[0] if (many >= 2).any() else s.uid.iloc[0]
    print("\nStep 2 example: one member's receipts (a gap > W starts a new subscription)")
    e = ev[ev.uid == ex_uid].copy(); e["gap_days"] = e.dt.diff().dt.days
    display(e[["dt", "channel", "plan", "gap_days", "W", "new_sub", "sub_id"]].head(example_rows))
    print("...and the subscriptions built from them (detect = last payment + W; churned if detect <= data end)")
    display(s[s.uid == ex_uid][["start", "last_pay", "n_pay", "channel", "plan", "W", "detect", "churned"]])
    m = monthly(s).merge(published(provider), on="month", how="left")
    print("\nSteps 3-5 by month: simple recompute vs published subs_monthly (pub_*)")
    display(m.tail(15))
    print("Churn rates the model applied, last 3 months (rate_source = full_panel or reliable_subset):")
    display(rates(provider))
    return ev, s, m

### ChatGPT

Apple, Google Play and (EU) PayPal/Stripe. **Direct ChatGPT is excluded**: its sign-up receipts have no renewals, so churn can't be measured.

In [7]:
ev_chatgpt, s_chatgpt, m_chatgpt = walkthrough("chatgpt")

chatgpt: 1,030,781 paid receipts, 145,388 members, 219,328 subscriptions
Lapse window W (days) by channel x cadence: {('apple', 'A'): 400, ('apple', 'M'): 42, ('google_play', 'M'): 40}

Step 2 example: one member's receipts (a gap > W starts a new subscription)


,dt,channel,plan,gap_days,W,new_sub,sub_id
85,2024-01-29,apple,plus_M,NaN,42,True,28
86,2024-02-29,apple,plus_M,31.0,42,False,28
87,2024-03-29,apple,plus_M,29.0,42,False,28
88,2024-04-29,apple,plus_M,31.0,42,False,28
89,2024-10-31,apple,plus_M,185.0,42,True,29
90,2024-11-26,apple,plus_M,26.0,42,False,29
91,2024-12-26,apple,plus_M,30.0,42,False,29
92,2025-01-26,apple,plus_M,31.0,42,False,29
93,2025-02-26,apple,plus_M,31.0,42,False,29
94,2025-06-28,apple,plus_M,122.0,42,True,30


...and the subscriptions built from them (detect = last payment + W; churned if detect <= data end)


,start,last_pay,n_pay,channel,plan,W,detect,churned
sub_id,,,,,,,,
28,2024-01-29,2024-04-29,4,apple,plus_M,42,2024-06-10,True
29,2024-10-31,2025-02-26,5,apple,plus_M,42,2025-04-09,True
30,2025-06-28,2025-11-27,6,apple,plus_M,42,2026-01-08,True



Steps 3-5 by month: simple recompute vs published subs_monthly (pub_*)


,month,gross_adds,churn,live_start,live_end,churn_rate,pub_gross_adds,pub_churn,pub_churn_rate,pub_subs,pub_subs_observed,held
19,2025-08,10671,6970,37958,41659,0.1836,11495,6412.0,0.1387,51305.0,41659,False
20,2025-09,9133,7865,41659,42927,0.1888,9964,7662.0,0.1493,53607.0,42927,False
21,2025-10,8849,7369,42927,44407,0.1717,9663,7032.0,0.1312,56238.0,44407,False
22,2025-11,9675,7072,44407,47010,0.1593,10457,7076.0,0.1258,59619.0,47010,False
23,2025-12,8301,6732,47010,48579,0.1432,9111,6577.0,0.1103,62154.0,48579,False
24,2026-01,10120,7191,48579,51508,0.1480,11022,7411.0,0.1192,65764.0,51508,False
25,2026-02,10451,6398,51508,55561,0.1242,11310,6726.0,0.1023,70348.0,55561,False
26,2026-03,10772,8447,55561,57886,0.1520,11904,8659.0,0.1231,73593.0,57886,False
27,2026-04,11460,9256,57886,60090,0.1599,12618,9743.0,0.1324,76468.0,60090,False
28,2026-05,11748,8869,60090,62969,0.1476,12981,9279.0,0.1213,80170.0,62969,False


Churn rates the model applied, last 3 months (rate_source = full_panel or reliable_subset):


,month,channel,rate_source,active,churned,rate
0,2026-08,apple,reliable_subset,36348,3928,0.1081
1,2026-08,google_play,full_panel,12645,1670,0.1321
2,2026-09,apple,reliable_subset,35196,2668,0.0758
3,2026-09,google_play,full_panel,13594,1637,0.1204


### Claude

Mostly direct (Anthropic receipts at list price), plus Apple and Google Play. Direct and Google Play use full-panel churn rates.

In [8]:
ev_claude, s_claude, m_claude = walkthrough("claude")

claude: 472,420 paid receipts, 90,842 members, 116,095 subscriptions
Lapse window W (days) by channel x cadence: {('apple', 'A'): 400, ('apple', 'M'): 43, ('direct', 'A'): 400, ('direct', 'M'): 41, ('google_play', 'A'): 400, ('google_play', 'M'): 42}

Step 2 example: one member's receipts (a gap > W starts a new subscription)


,dt,channel,plan,gap_days,W,new_sub,sub_id
48,2026-04-24,direct,pro_M,NaN,41,True,18
49,2026-05-24,direct,pro_M,30.0,41,False,18
50,2026-06-24,direct,pro_M,31.0,41,False,18
51,2026-07-24,direct,pro_M,30.0,41,False,18
52,2026-09-18,direct,pro_M,56.0,41,True,19
53,2026-09-30,direct,pro_M,12.0,41,False,19
54,2026-10-19,direct,pro_M,19.0,41,False,19


...and the subscriptions built from them (detect = last payment + W; churned if detect <= data end)


,start,last_pay,n_pay,channel,plan,W,detect,churned
sub_id,,,,,,,,
18,2026-04-24,2026-07-24,4,direct,pro_M,41,2026-09-03,True
19,2026-09-18,2026-10-19,3,direct,pro_M,41,2026-11-29,False



Steps 3-5 by month: simple recompute vs published subs_monthly (pub_*)


,month,gross_adds,churn,live_start,live_end,churn_rate,pub_gross_adds,pub_churn,pub_churn_rate,pub_subs,pub_subs_observed,held
19,2025-08,1501,861,6476,7116,0.1330,1617,962.0,0.1489,7116.0,7118,False
20,2025-09,1415,994,7116,7537,0.1397,1546,1111.0,0.1562,7551.0,7542,False
21,2025-10,1882,1051,7537,8368,0.1394,2006,1169.0,0.1548,8388.0,8373,False
22,2025-11,1873,1024,8368,9217,0.1224,2028,1167.0,0.1392,9249.0,9224,False
23,2025-12,1919,1213,9217,9923,0.1316,2068,1350.0,0.1460,9967.0,9929,False
24,2026-01,3489,1371,9923,12041,0.1382,3671,1546.0,0.1551,12092.0,12044,False
25,2026-02,7021,1247,12041,17815,0.1036,7289,1502.0,0.1242,17879.0,17818,False
26,2026-03,15248,1882,17815,31181,0.1056,15637,2181.0,0.1220,31335.0,31186,False
27,2026-04,16150,3284,31181,44047,0.1053,16793,3683.0,0.1175,44445.0,44057,False
28,2026-05,11794,5457,44047,50384,0.1239,12624,5805.0,0.1306,51264.0,50403,False


Churn rates the model applied, last 3 months (rate_source = full_panel or reliable_subset):


,month,channel,rate_source,active,churned,rate
0,2026-08,apple,reliable_subset,7080,848,0.1198
1,2026-08,direct,full_panel,46258,4791,0.1036
2,2026-08,google_play,full_panel,1825,283,0.1551
3,2026-09,apple,reliable_subset,7364,831,0.1128
4,2026-09,direct,full_panel,47757,4942,0.1035
5,2026-09,google_play,full_panel,1912,298,0.1559


### Gemini

Google Play (including Google One web purchases) and Apple; AI tiers only. Storage-only plans are excluded; AI Plus 2 TB is included.

In [9]:
ev_gemini, s_gemini, m_gemini = walkthrough("gemini")

gemini: 528,093 paid receipts, 91,042 members, 108,497 subscriptions
Lapse window W (days) by channel x cadence: {('apple', 'A'): 400, ('apple', 'M'): 44, ('google_play', 'A'): 400, ('google_play', 'M'): 40}

Step 2 example: one member's receipts (a gap > W starts a new subscription)


,dt,channel,plan,gap_days,W,new_sub,sub_id
106,2025-01-12,google_play,ai_pro_M,NaN,40,True,23
107,2025-02-12,google_play,ai_pro_M,31.0,40,False,23
108,2025-03-12,google_play,ai_pro_M,28.0,40,False,23
109,2025-04-12,google_play,ai_pro_M,31.0,40,False,23
110,2025-05-12,google_play,ai_pro_M,30.0,40,False,23
111,2026-02-01,google_play,ai_plus_2tb_M,265.0,40,True,24
112,2026-03-01,google_play,ai_plus_2tb_M,28.0,40,False,24
113,2026-04-01,google_play,ai_plus_2tb_M,31.0,40,False,24
114,2026-05-01,google_play,ai_plus_2tb_M,30.0,40,False,24
115,2026-06-01,google_play,ai_plus_2tb_M,31.0,40,False,24


...and the subscriptions built from them (detect = last payment + W; churned if detect <= data end)


,start,last_pay,n_pay,channel,plan,W,detect,churned
sub_id,,,,,,,,
23,2025-01-12,2025-05-12,5,google_play,ai_pro_M,40,2025-06-21,True
24,2026-02-01,2026-10-01,9,google_play,ai_plus_2tb_M,40,2026-11-10,False



Steps 3-5 by month: simple recompute vs published subs_monthly (pub_*)


,month,gross_adds,churn,live_start,live_end,churn_rate,pub_gross_adds,pub_churn,pub_churn_rate,pub_subs,pub_subs_observed,held
19,2025-08,1498,1157,8116,8457,0.1426,1627,1279.0,0.1572,8486.0,8458,False
20,2025-09,1567,1047,8457,8977,0.1238,1717,1195.0,0.1408,9008.0,8978,False
21,2025-10,1669,1010,8977,9636,0.1125,1823,1156.0,0.1283,9675.0,9640,False
22,2025-11,1935,1056,9636,10515,0.1096,2069,1193.0,0.1233,10552.0,10518,False
23,2025-12,2755,1014,10515,12256,0.0964,2912,1166.0,0.1105,12298.0,12259,False
24,2026-01,6836,1281,12256,17811,0.1045,7046,1482.0,0.1205,17862.0,17814,False
25,2026-02,23070,1296,17811,39585,0.0728,23351,1552.0,0.0869,39661.0,39591,False
26,2026-03,6049,2189,39585,43445,0.0553,6599,2734.0,0.0689,43526.0,43454,False
27,2026-04,5590,3137,43445,45898,0.0722,6197,3713.0,0.0853,46010.0,45909,False
28,2026-05,6054,3080,45898,48872,0.0671,6589,3591.0,0.0781,49008.0,48887,False


Churn rates the model applied, last 3 months (rate_source = full_panel or reliable_subset):


,month,channel,rate_source,active,churned,rate
0,2026-08,apple,reliable_subset,1565,275,0.1757
1,2026-08,google_play,full_panel,54406,3892,0.0715
2,2026-09,apple,reliable_subset,1495,150,0.1003
3,2026-09,google_play,full_panel,58723,4170,0.0710


### Grok (incl. X Premium+)

Apple and Google Play SuperGrok/Lite/Heavy, plus X Premium+. There are no direct xAI receipts. Long lapse windows (60–90 days) because renewals arrive irregularly.

In [10]:
ev_grok, s_grok, m_grok = walkthrough("grok")

grok: 49,212 paid receipts, 14,636 members, 17,692 subscriptions
Lapse window W (days) by channel x cadence: {('apple', 'A'): 400, ('apple', 'M'): 60, ('apple', 'Q'): 135, ('google_play', 'A'): 400, ('google_play', 'M'): 91}

Step 2 example: one member's receipts (a gap > W starts a new subscription)


,dt,channel,plan,gap_days,W,new_sub,sub_id
14,2023-12-04,apple,premium_plus_M,NaN,60,True,13
15,2024-01-03,apple,premium_plus_M,30.0,60,False,13
16,2024-02-03,apple,premium_plus_M,31.0,60,False,13
17,2025-02-23,apple,premium_plus_M,386.0,60,True,14
18,2025-02-24,apple,supergrok_M,1.0,60,False,14
19,2025-04-15,apple,supergrok_M,50.0,60,False,14


...and the subscriptions built from them (detect = last payment + W; churned if detect <= data end)


,start,last_pay,n_pay,channel,plan,W,detect,churned
sub_id,,,,,,,,
13,2023-12-04,2024-02-03,3,apple,premium_plus_M,60,2024-04-03,True
14,2025-02-23,2025-04-15,3,apple,premium_plus_M,60,2025-06-14,True



Steps 3-5 by month: simple recompute vs published subs_monthly (pub_*)


,month,gross_adds,churn,live_start,live_end,churn_rate,pub_gross_adds,pub_churn,pub_churn_rate,pub_subs,pub_subs_observed,held
19,2025-08,545,124,1562,1983,0.0794,579,155.0,0.0915,2119.0,1983,False
20,2025-09,230,423,1983,1790,0.2133,254,401.0,0.1892,1972.0,1790,False
21,2025-10,473,437,1790,1826,0.2441,503,495.0,0.2511,1980.0,1826,False
22,2025-11,615,275,1826,2166,0.1506,680,315.0,0.1593,2345.0,2166,False
23,2025-12,690,357,2166,2499,0.1648,756,378.0,0.1612,2723.0,2499,False
24,2026-01,1046,497,2499,3048,0.1989,1120,507.0,0.1860,3336.0,3049,False
25,2026-02,699,434,3048,3313,0.1424,802,466.0,0.1396,3672.0,3314,False
26,2026-03,1361,747,3313,3927,0.2255,1483,802.0,0.2183,4354.0,3929,False
27,2026-04,1625,605,3927,4947,0.1541,1787,677.0,0.1555,5464.0,4949,False
28,2026-05,1227,817,4947,5357,0.1652,1408,957.0,0.1752,5915.0,5358,False


Churn rates the model applied, last 3 months (rate_source = full_panel or reliable_subset):


,month,channel,rate_source,active,churned,rate
0,2026-08,apple,reliable_subset,3000,536,0.1787
1,2026-08,google_play,reliable_subset,526,92,0.1749
2,2026-09,apple,reliable_subset,3026,536,0.1771
3,2026-09,google_play,reliable_subset,587,102,0.1738


### Perplexity

Apple and Google Play only. Direct Perplexity sends unpriced welcome emails, not paid receipts, so it is not counted.

In [11]:
ev_perplexity, s_perplexity, m_perplexity = walkthrough("perplexity")

perplexity: 21,051 paid receipts, 5,526 members, 6,319 subscriptions
Lapse window W (days) by channel x cadence: {('apple', 'A'): 400, ('apple', 'M'): 59, ('google_play', 'A'): 400, ('google_play', 'M'): 58, ('other', 'A'): 400, ('other', 'M'): 62}

Step 2 example: one member's receipts (a gap > W starts a new subscription)


,dt,channel,plan,gap_days,W,new_sub,sub_id
16,2024-11-17,google_play,pro_M,NaN,58,True,9
17,2024-12-17,google_play,pro_M,30.0,58,False,9
18,2025-01-17,google_play,pro_M,31.0,58,False,9
19,2025-02-17,google_play,pro_M,31.0,58,False,9
20,2025-03-17,google_play,pro_M,28.0,58,False,9
21,2025-04-17,google_play,pro_M,31.0,58,False,9
22,2025-05-17,google_play,pro_M,30.0,58,False,9
23,2025-06-17,google_play,pro_M,31.0,58,False,9
24,2025-07-17,google_play,pro_M,30.0,58,False,9
25,2025-08-17,google_play,pro_M,31.0,58,False,9


...and the subscriptions built from them (detect = last payment + W; churned if detect <= data end)


,start,last_pay,n_pay,channel,plan,W,detect,churned
sub_id,,,,,,,,
9,2024-11-17,2025-08-17,10,google_play,pro_M,58,2025-10-14,True
10,2026-05-04,2026-10-04,6,google_play,pro_M,58,2026-12-01,False



Steps 3-5 by month: simple recompute vs published subs_monthly (pub_*)


,month,gross_adds,churn,live_start,live_end,churn_rate,pub_gross_adds,pub_churn,pub_churn_rate,pub_subs,pub_subs_observed,held
19,2025-08,133,108,812,837,0.1330,143,144.0,0.1660,868.0,838,False
20,2025-09,164,119,837,882,0.1422,177,127.0,0.1467,917.0,883,False
21,2025-10,208,111,882,979,0.1259,223,160.0,0.1739,981.0,980,False
22,2025-11,195,144,979,1030,0.1471,206,135.0,0.1377,1052.0,1031,False
23,2025-12,194,139,1030,1085,0.1350,206,144.0,0.1369,1114.0,1086,False
24,2026-01,255,159,1085,1181,0.1465,267,162.0,0.1451,1219.0,1182,False
25,2026-02,288,139,1181,1330,0.1177,307,154.0,0.1260,1373.0,1331,False
26,2026-03,441,200,1330,1571,0.1504,471,240.0,0.1745,1604.0,1572,False
27,2026-04,533,216,1571,1888,0.1375,568,233.0,0.1452,1939.0,1890,False
28,2026-05,580,241,1888,2227,0.1276,621,273.0,0.1405,2288.0,2229,False


Churn rates the model applied, last 3 months (rate_source = full_panel or reliable_subset):


,month,channel,rate_source,active,churned,rate
0,2026-08,apple,reliable_subset,1453,262,0.1803
1,2026-08,google_play,reliable_subset,101,12,0.1188
2,2026-08,other,reliable_subset,9,2,0.2222
3,2026-09,apple,reliable_subset,1384,214,0.1546
4,2026-09,google_play,reliable_subset,102,17,0.1667
5,2026-09,other,reliable_subset,10,0,0.0000


### Muse

Apple 'Power Plan' and Google Play since 7 Sep 2026. Churn becomes observable once the first cohorts pass their window (late October 2026).

In [12]:
ev_muse, s_muse, m_muse = walkthrough("muse")

muse: 464 paid receipts, 440 members, 440 subscriptions
Lapse window W (days) by channel x cadence: {('apple', 'M'): 62, ('google_play', 'M'): 62}

Step 2 example: one member's receipts (a gap > W starts a new subscription)


,dt,channel,plan,gap_days,W,new_sub,sub_id
0,2026-10-02,google_play,power_M,NaN,62,True,1


...and the subscriptions built from them (detect = last payment + W; churned if detect <= data end)


,start,last_pay,n_pay,channel,plan,W,detect,churned
sub_id,,,,,,,,
1,2026-10-02,2026-10-02,1,google_play,power_M,62,2026-12-03,False



Steps 3-5 by month: simple recompute vs published subs_monthly (pub_*)


,month,gross_adds,churn,live_start,live_end,churn_rate,pub_gross_adds,pub_churn,pub_churn_rate,pub_subs,pub_subs_observed,held
19,2025-08,0,0,0,0,NaN,0,0.0,NaN,0.0,0,False
20,2025-09,0,0,0,0,NaN,0,0.0,NaN,0.0,0,False
21,2025-10,0,0,0,0,NaN,0,0.0,NaN,0.0,0,False
22,2025-11,0,0,0,0,NaN,0,0.0,NaN,0.0,0,False
23,2025-12,0,0,0,0,NaN,0,0.0,NaN,0.0,0,False
24,2026-01,0,0,0,0,NaN,0,0.0,NaN,0.0,0,False
25,2026-02,0,0,0,0,NaN,0,0.0,NaN,0.0,0,False
26,2026-03,0,0,0,0,NaN,0,0.0,NaN,0.0,0,False
27,2026-04,0,0,0,0,NaN,0,0.0,NaN,0.0,0,False
28,2026-05,0,0,0,0,NaN,0,0.0,NaN,0.0,0,False


Churn rates the model applied, last 3 months (rate_source = full_panel or reliable_subset):


,month,channel,rate_source,active,churned,rate


## 5. Subscribers: the stock-flow index

The published subscriber count (`subs`) follows a monthly identity:

> subs(t) = subs(t−1) + gross adds(t) − churn(t)

Churn(t) is the sum over plans and tenure bands of (churn rate × subscribers), using the rates in `churn_rates`. The check below confirms the published table adds up.

`subs_observed` is the simple open-subscription count from the full panel, shown for comparison. The model sits above it mainly on Apple, where missed renewal emails make some live subscribers look churned.

In [13]:
pub = q(f'''
SELECT provider, TO_VARCHAR(month, 'YYYY-MM') AS month, subs, gross_adds, churn, subs_observed, share, held
FROM srs_mcp.ai_consumer.subs_monthly
WHERE region = '{REGION}' AND country = '{COUNTRY}' AND plan = 'all' ORDER BY 1, 2''')
pub["identity_gap"] = (pub.subs - (pub.groupby("provider").subs.shift() + pub.gross_adds - pub.churn)).round(1)
print("Largest absolute identity gap by provider, excluding held months (should be ~0):")
display(pub[~pub.held].groupby("provider").identity_gap.apply(lambda x: x.abs().max()))
latest = pub[~pub.held].month.max()
print("Latest non-held month:", latest)
pub[pub.month == latest][["provider", "subs", "subs_observed", "share", "gross_adds", "churn"]]

Largest absolute identity gap by provider, excluding held months (should be ~0):


provider
chatgpt       0.0
claude        0.0
gemini        0.0
grok          0.0
muse          0.0
perplexity    0.0
Name: identity_gap, dtype: float64

Latest non-held month: 2026-09


,provider,subs,subs_observed,share,gross_adds,churn
32,chatgpt,86307.417193,65670,0.385978,9392,6608.730891
66,claude,62406.178053,60593,0.279088,8072,6574.651541
100,gemini,65782.850474,65599,0.294189,9081,4433.902728
134,grok,6774.971461,6170,0.030299,1573,1161.980394
168,muse,133.000000,133,0.000595,133,0.000000
202,perplexity,2202.955018,2096,0.009852,316,371.371828
